<a href="https://colab.research.google.com/github/caochengrui/DQN/blob/main/DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Network (DQN)



### Install Dependencies

In [ ]:
!pip install git+https://github.com/caochengrui/DQN --upgrade

  Cloning https://github.com/caochengrui/DQN to /tmp/pip-req-build-5qf6jvnl
  Running command git clone --filter=blob:none --quiet https://github.com/caochengrui/DQN /tmp/pip-req-build-5qf6jvnl
  Resolved https://github.com/caochengrui/DQN to commit e1e7d057738c6f9a7b7ebcca3fc99bff6536ab41
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 24.8 MB/s eta 0:00:00
  Created wheel for DQN: filename=dqn-0.1.dev1+ge1e7d0577-py3-none-any.whl size=7805 sha256=71137489342923d150f2f293397086d08e341fd9a3830f20eb1096a92aa93317
  Stored in directory: /tmp/pip-ephem-wheel-cache-kld14xzv/wheels/a2/90/8d/5cd4aa3530bd4d08a5f9e96d13734318daa3d5e98cb81f7568
Successfully built DQN
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.3
    Uninstalling gymnasium-1.2.3:
      Successfully uninstalled gymnasium-1.2.3


In [ ]:
!apt-get install ffmpeg  # For visualization

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


### Imports

In [ ]:
from typing import Optional
import os

import numpy as np
import torch as th
import gymnasium as gym
from gymnasium import spaces

from DQN import ReplayBuffer, epsilon_greedy_action_selection, collect_one_step, linear_schedule, QNetwork
from DQN.evaluation import evaluate_policy
from custom_utils import notebook_show_videos

##  DQN Target Network


<div>
    <img src="attachment:c176b66c-2fc7-4e1e-a351-87ef832bdfac.png" width="1000"/>
</div>

The only things that is changing is when predicting the next q value.

In DQN without target, the online network with weights **$\theta$** is used:

$y = r_t + \gamma \cdot \max_{a \in A}(\hat{Q}_{\pi}(s_{t+1}, a; \theta))$


whereas with DQN with target network, the target q-network (a delayed copy of the q-network) with weights **$\theta^\prime$** is used instead:

$y = r_t + \gamma \cdot \max_{a \in A}(\hat{Q}_{\pi}(s_{t+1}, a; \theta^\prime))$


### DQN update with target network

In [ ]:
def dqn_update(
    q_net: QNetwork,
    q_target_net: QNetwork,
    optimizer: th.optim.Optimizer,
    replay_buffer: ReplayBuffer,
    batch_size: int,
    gamma: float,
) -> None:
    """
    Perform one gradient step on the Q-network
    using the data from the replay buffer.

    :param q_net: The Q-network to update
    :param q_target_net: The target Q-network, to compute the td-target.
    :param optimizer: The optimizer to use
    :param replay_buffer: The replay buffer containing the transitions
    :param batch_size: The minibatch size, how many transitions to sample
    :param gamma: The discount factor
    """

    replay_data = replay_buffer.sample(batch_size).to_torch()

    with th.no_grad():
        next_q_values = q_target_net(replay_data.next_observations)
        next_q_values, _ = next_q_values.max(dim=1)
        # If the episode is terminated, set the target to the reward
        should_bootstrap = th.logical_not(replay_data.terminateds)
        td_target = replay_data.rewards + gamma * next_q_values * should_bootstrap

    q_values = q_net(replay_data.observations)
    # Select the Q-values corresponding to the actions that were selected
    # during data collection
    current_q_values = th.gather(q_values, dim=1, index=replay_data.actions)
    # Reshape from (batch_size, 1) to (batch_size,) to avoid broadcast error
    current_q_values = current_q_values.squeeze(dim=1)

    assert current_q_values.shape == (batch_size,), f"{current_q_values.shape} != {(batch_size,)}"
    assert current_q_values.shape == td_target.shape, f"{current_q_values.shape} != {td_target.shape}"

    # Compute the Mean Squared Error (MSE) loss
    # Optionally, one can use a Huber loss instead of the MSE loss
    loss = ((current_q_values - td_target) ** 2).mean()
    # Huber loss
    # loss = th.nn.functional.smooth_l1_loss(current_q_values, td_target)


    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

### Updated training loop

In [ ]:
def run_dqn(
    env_id: str = "CartPole-v1",
    replay_buffer_size: int = 50_000,
    target_network_update_interval: int = 1000,
    # Warmup phase
    learning_starts: int = 100,
    exploration_initial_eps: float = 1.0,
    exploration_final_eps: float = 0.01,
    exploration_fraction: float = 0.1,
    n_timesteps: int = 20_000,
    update_interval: int = 2,
    learning_rate: float = 3e-4,
    batch_size: int = 64,
    gamma: float = 0.99,
    n_hidden_units: int = 64,
    n_eval_episodes: int = 10,
    evaluation_interval: int = 1000,
    eval_exploration_rate: float = 0.0,
    seed: int = 2026,
    eval_render_mode: Optional[str] = None,  # "human", "rgb_array", None
) -> QNetwork:
    """
    Run Deep Q-Learning (DQN) on a given environment.
    (with a target network)

    :param env_id: Name of the environment
    :param replay_buffer_size: Max capacity of the replay buffer
    :param target_network_update_interval: How often do we copy the parameters
         to the target network
    :param learning_starts: Warmup phase to fill the replay buffer
        before starting the optimization.
    :param exploration_initial_eps: The initial exploration rate
    :param exploration_final_eps: The final exploration rate
    :param exploration_fraction: The fraction of the number of steps
        during which the exploration rate is annealed from
        initial_eps to final_eps.
        After this many steps, the exploration rate remains constant.
    :param n_timesteps: Number of timesteps in total
    :param update_interval: How often to update the Q-network
        (every update_interval steps)
    :param learning_rate: The learning rate to use for the optimizer
    :param batch_size: The minibatch size
    :param gamma: The discount factor
    :param n_hidden_units: Number of units for each hidden layer
        of the Q-Network.
    :param n_eval_episodes: The number of episodes to evaluate the policy on
    :param evaluation_interval: How often to evaluate the policy
    :param eval_exploration_rate: The exploration rate to use during evaluation
    :param seed: Random seed for the pseudo random generator
    :param eval_render_mode: The render mode to use for evaluation
    """

    np.random.seed(seed)
    th.manual_seed(seed)

    os.makedirs("./logs/", exist_ok=True)
    os.makedirs("./logs/checkpoint/", exist_ok=True)

    env = gym.make(env_id)
    # For highway env
    env = gym.wrappers.FlattenObservation(env)
    env = gym.wrappers.RecordEpisodeStatistics(env)
    assert isinstance(env.observation_space, spaces.Box)
    assert isinstance(env.action_space, spaces.Discrete)
    env.action_space.seed(seed)

    eval_env = gym.make(env_id, render_mode=eval_render_mode)
    eval_env = gym.wrappers.FlattenObservation(eval_env)
    eval_env.reset(seed=seed)
    eval_env.action_space.seed(seed)


    q_net = QNetwork(env.observation_space, env.action_space, n_hidden_units=n_hidden_units)
    q_target_net = QNetwork(env.observation_space, env.action_space, n_hidden_units=n_hidden_units)
    q_target_net.load_state_dict(q_net.state_dict())

    # For flappy bird
    if env.observation_space.dtype == np.float64:
        q_net.double()
        q_target_net.double()

    optimizer = th.optim.Adam(q_net.parameters(), lr=learning_rate)

    replay_buffer = ReplayBuffer(replay_buffer_size, env.observation_space, env.action_space)
    obs, _ = env.reset(seed=seed)
    for current_step in range(1, n_timesteps + 1):
        exploration_rate = linear_schedule(
            exploration_initial_eps,
            exploration_final_eps,
            current_step,
            int(exploration_fraction * n_timesteps),
        )

        obs = collect_one_step(
            env,
            q_net,
            replay_buffer,
            obs,
            exploration_rate=exploration_rate,
            verbose=0,
        )

        if (current_step % target_network_update_interval) == 0:
            q_target_net.load_state_dict(q_net.state_dict())

        if (current_step % update_interval) == 0 and current_step > learning_starts:
            dqn_update(q_net, q_target_net, optimizer, replay_buffer, batch_size, gamma=gamma)

        if (current_step % evaluation_interval) == 0:
            print()
            print(f"Evaluation at step {current_step}:")
            evaluate_policy(eval_env, q_net, n_eval_episodes, eval_exploration_rate=eval_exploration_rate)
            th.save(q_net.state_dict(), f"./logs/checkpoint/q_net_checkpoint_{env_id}_{current_step}.pth")
    return q_net

## Train DQN agent with target network on CartPole env

In [ ]:
# Tuned hyperparameters from the RL Zoo3 of the Stable Baselines3 library
# https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml

env_id = "CartPole-v1"

q_net = run_dqn(
    env_id=env_id,
    replay_buffer_size=100_000,
    target_network_update_interval=10,
    learning_starts=1000,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.04,
    exploration_fraction=0.1,
    n_timesteps=80_000,
    update_interval=2,
    learning_rate=1e-3,
    batch_size=64,
    gamma=0.99,
    n_eval_episodes=10,
    evaluation_interval=5000,
    eval_exploration_rate=0.0,
    seed=2026,
)


Evaluation at step 5000:
Mean episode reward: 236.20 +/- 65.39

Evaluation at step 10000:
Mean episode reward: 268.80 +/- 110.91

Evaluation at step 15000:
Mean episode reward: 251.80 +/- 74.17

Evaluation at step 20000:
Mean episode reward: 480.30 +/- 59.10

Evaluation at step 25000:
Mean episode reward: 334.00 +/- 109.74

Evaluation at step 30000:
Mean episode reward: 487.20 +/- 38.40

Evaluation at step 35000:
Mean episode reward: 432.10 +/- 100.33

Evaluation at step 40000:
Mean episode reward: 332.50 +/- 44.06

Evaluation at step 45000:
Mean episode reward: 255.30 +/- 21.19

Evaluation at step 50000:
Mean episode reward: 247.30 +/- 34.22

Evaluation at step 55000:
Mean episode reward: 211.70 +/- 10.81

Evaluation at step 60000:
Mean episode reward: 217.70 +/- 12.88

Evaluation at step 65000:
Mean episode reward: 478.00 +/- 39.90

Evaluation at step 70000:
Mean episode reward: 356.80 +/- 21.12

Evaluation at step 75000:
Mean episode reward: 500.00 +/- 0.00

Evaluation at step 8000

### Visualize the trained agent

In [ ]:
eval_env = gym.make(env_id, render_mode="rgb_array")
n_eval_episodes = 3
eval_exploration_rate = 0.0
video_name = f"DQN_{env_id}"

q_net = QNetwork(eval_env.observation_space, eval_env.action_space, n_hidden_units=64)
q_net.load_state_dict(th.load("./logs/checkpoint/q_net_checkpoint_CartPole-v1_75000.pth"))

evaluate_policy(
    eval_env,
    q_net,
    n_eval_episodes,
    eval_exploration_rate=eval_exploration_rate,
    video_name=video_name,
)

notebook_show_videos("./logs/videos/", prefix=video_name)

Saving video to logs/videos/DQN_CartPole-v1.mp4
Mean episode reward: 500.00 +/- 0.00


## Training DQN agent on flappy bird:

You can go in the [GitHub repo](https://github.com/araffin/flappy-bird-gymnasium/tree/patch-1) to learn more about this environment.

<div>
    <img src="https://raw.githubusercontent.com/markub3327/flappy-bird-gymnasium/main/imgs/dqn.gif" width="300"/>
</div>


In [ ]:
!pip install "flappy-bird-gymnasium @ git+https://github.com/araffin/flappy-bird-gymnasium@patch-1"

  Cloning https://github.com/araffin/flappy-bird-gymnasium (to revision patch-1) to /tmp/pip-install-yoqmvgd0/flappy-bird-gymnasium_040c3a799f4f4199b4144179e25416de
  Running command git clone --filter=blob:none --quiet https://github.com/araffin/flappy-bird-gymnasium /tmp/pip-install-yoqmvgd0/flappy-bird-gymnasium_040c3a799f4f4199b4144179e25416de
  Running command git checkout -b patch-1 --track origin/patch-1
  Switched to a new branch 'patch-1'
  Branch 'patch-1' set up to track remote branch 'patch-1' from 'origin'.
  Resolved https://github.com/araffin/flappy-bird-gymnasium to commit 8828737242a38c0cd18c4260819fe7e695fb6bae
  Preparing metadata (setup.py) ... done
  Created wheel for flappy-bird-gymnasium: filename=flappy_bird_gymnasium-0.2.2-py3-none-any.whl size=1074471 sha256=5066f28499d2712119b312cb6beda2947771a47c9f42858d636091aee23a1603
  Stored in directory: /tmp/pip-ephem-wheel-cache-_sj8wh1l/wheels/b0/48/e5/e447569a935d86b16cf1fd7560c83ae96ea0ebaaec403a04f2
Successfully b

In [ ]:
import flappy_bird_gymnasium  # noqa: F401

In [ ]:
env_id = "FlappyBird-v0"

q_net = run_dqn(
    env_id=env_id,
    replay_buffer_size=100_000,
    target_network_update_interval=250,
    learning_starts=10_000,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.03,
    exploration_fraction=0.1,
    n_timesteps=500_000,
    update_interval=4,
    learning_rate=1e-3,
    batch_size=128,
    gamma=0.98,
    n_eval_episodes=5,
    evaluation_interval=50000,
    n_hidden_units=256,
    eval_exploration_rate=0.0,
    seed=2026,
    eval_render_mode=None,
)


Evaluation at step 50000:
Mean episode reward: 9.14 +/- 0.17

Evaluation at step 100000:
Mean episode reward: 14.40 +/- 1.80

Evaluation at step 150000:
Mean episode reward: 17.24 +/- 5.41

Evaluation at step 200000:
Mean episode reward: 49.80 +/- 48.23

Evaluation at step 250000:
Mean episode reward: 152.16 +/- 108.85

Evaluation at step 300000:
Mean episode reward: 213.20 +/- 141.06

Evaluation at step 350000:
Mean episode reward: 923.40 +/- 1116.54

Evaluation at step 400000:
Mean episode reward: 111.12 +/- 101.24

Evaluation at step 450000:
Mean episode reward: 1397.28 +/- 1816.98

Evaluation at step 500000:
Mean episode reward: 481.90 +/- 377.90


### Record a video of the trained agent

In [ ]:
eval_env = gym.make(env_id, render_mode="rgb_array")
n_eval_episodes = 3
eval_exploration_rate = 0.00
video_name = f"DQN_{env_id}"
q_net = QNetwork(eval_env.observation_space, eval_env.action_space, n_hidden_units=256)
# Convert weights from float32 to float64 to match flappy bird obs
q_net.double()
q_net.load_state_dict(th.load("./logs/checkpoint/q_net_checkpoint_FlappyBird-v0_500000.pth"))

evaluate_policy(
    eval_env,
    q_net,
    n_eval_episodes,
    eval_exploration_rate=eval_exploration_rate,
    video_name=video_name,
)

notebook_show_videos("./logs/videos/", prefix=video_name)

Saving video to logs/videos/DQN_FlappyBird-v0.mp4
Mean episode reward: 179.83 +/- 133.71


### Going further

- analyse the learned q-values
- explore different value for the target update, use soft update instead of hard-copy
- experiment with Huber loss (smooth l1 loss) instead of l2 loss (mean squared error)
- play with different environments
- implement a CNN to play flappybird/pong from pixels (need to stack frames)
- implement DQN extensions (double Q-learning, prioritized experience replay, ...)